# 02 - LoRA on SAM
Harness for the LoRA adaptation arm:
sanity check the LoRALayer, then load SAM, inject LoRA into the encoder and visualize predictions

In [5]:
#autoreload, so edits to .py files show without restarting
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys; sys.path.insert(0, "../src")
import torch
from medseg.models.lora import LoRALayer

## 1. Sanity-check for LoRALayer

In [7]:
!pytest ../tests/test_lora.py -v

7596.20s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


============================= test session starts ==============================
platform darwin -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- /Users/nikhilclaude/anaconda3/envs/sam-medseg-mac/bin/python
cachedir: .pytest_cache
rootdir: /Users/nikhilclaude/Documents/Projects/DeepLearning/group_project
plugins: anyio-4.14.2
collected 3 items                                                              

../tests/test_lora.py::test_output_shape PASSED                          [ 33%]
../tests/test_lora.py::test_zero_at_init PASSED                          [ 66%]
../tests/test_lora.py::test_param_count PASSED                           [100%]

============================== 3 passed in 0.65s ===============================


In [8]:
import sys; sys.path.insert(0, "../src")
import torch

from segment_anything import sam_model_registry
from medseg.models.sam_lora import inject_lora_into_sam

sam = sam_model_registry["vit_b"](checkpoint="../checkpoints/sam_vit_b_01ec64.pth")

def counts(m):
    total = sum(p.numel() for p in m.parameters())
    train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

t, tr = counts(sam)
print(f"Before LoRA:  {tr:,} / {t:,} trainable  ({100*tr/t:.2f}%)")


inject_lora_into_sam(sam,r=4, alpha=4)

t, tr = counts(sam)
print(f"After  LoRA:  {tr:,} / {t:,} trainable  ({100*tr/t:.4f}%)")



Before LoRA:  93,735,472 / 93,735,472 trainable  (100.00%)
After  LoRA:  147,456 / 93,882,928 trainable  (0.1571%)
